# Python Core Refresher — Capstone Project Workbook

**Two industry-style projects, one workbook.** Each project is written the way a junior engineer
would actually receive it on the job: a problem statement, a PRD (Product Requirements Document),
a simple architecture, and the team's coding standards — not a tutorial.

| | |
|---|---|
| **Projects included** | 1. Telecom CDR & Billing Insights CLI · 2. Retail Inventory & Sales Reconciliation CLI |
| **Skills exercised** | Setup, syntax, variables & data types, control flow & loops · Data structures (list/dict/set), functions & basic OOP, string manipulation · File handling (CSV/JSON), scripting best practices |
| **Solutions included?** | No — this is a workbook, not a lesson. Build it yourself against the acceptance criteria. |
| **Format** | Each project is broken into 3 milestones, mirroring the 3 Python Core Refresher sessions. Each milestone is a set of numbered, problem-statement-style questions with an empty code cell for your implementation. |

### How to use this workbook

1. Read the PRD and Architecture for a project fully before writing any code — in a real job, skipping
   this step is how people build the wrong thing correctly.
2. Set up the folder structure described in the Architecture section first. Don't write everything
   into one file, even though it's tempting for something this size.
3. Work through the milestones in order — each one deliberately depends on functions/classes you
   built in the previous milestone.
4. Test against the provided sample data (generated by the code cells in each project) before
   considering a milestone done.
5. Before submitting, run through the Deliverables Checklist and self-score against the Evaluation
   Rubric at the end of this workbook.


---
## Engineering Coding Standards (Applies to Both Projects)

Every project in this workbook is graded against these standards, not just "does it run":

1. **Style** — PEP8. `snake_case` for functions/variables, `PascalCase` for classes, `UPPER_CASE`
   for constants.
2. **Docstrings** — every function/class gets a docstring: one-line summary, then `Args:` /
   `Returns:` if it takes or returns anything non-obvious.
3. **No bare `except:`** — catch specific exception types. If you genuinely don't know what could go
   wrong, that's a sign you haven't thought through the failure mode yet, not a reason to catch everything.
4. **No `print()` for status/errors** — use the `logging` module. `print()` is reserved for the final,
   intentional console summary a user is meant to read.
5. **No top-level side effects in a module** — nothing should run just from `import`-ing a file. Guard
   script entry points with `if __name__ == "__main__":`.
6. **Constants live in one place** — a `config.py` (or a clearly-named constants block at the top of
   one module), never scattered magic numbers across files.
7. **Modular structure** — the Architecture section for each project describes the expected module
   breakdown. A single 300-line script that does everything is a rewrite, not a passing submission.
8. **Type hints on function signatures** — not enforced by a type checker here, but expected as
   documentation, e.g. `def compute_cost(call_type: str, duration_sec: int) -> float:`.


---
# Project 1 — Telecom CDR & Billing Insights CLI

**Client:** Orbit Mobile Pvt Ltd (fictitious regional telecom operator)
**Requested by:** Billing Operations, via the Engineering Lead


## 1.1 Problem Statement / PRD

**Background.** Orbit Mobile's billing ops team currently reconciles each day's call records by
hand in a spreadsheet — pulling a raw call log, cross-referencing subscriber plans, calculating
charges, and flagging anything unusual. As call volume has grown, this has become slow and
error-prone. Engineering has scoped a small internal CLI tool to automate this for a single day's
batch. You are the engineer assigned to build it.

**Goals**
- Automate ingestion and validation of a day's CDR (Call Detail Record) batch.
- Compute per-subscriber billing based on call type and duration.
- Flag suspicious call patterns for manual fraud review.
- Produce a daily summary report, both machine-readable (JSON) and human-readable (console).

**Out of scope (Phase 2, not this project)**
- No live network/database integration — file-based batch processing only.
- No multi-day historical trend analysis.
- No user interface — this is a CLI tool run by ops/engineering, not customer-facing.

**Stakeholders**
- *Billing Ops Analyst* — consumes the daily report, needs it accurate and easy to read.
- *Engineering Lead* — reviews code quality and maintainability before this gets scheduled as a cron job.

### Functional Requirements

| ID | Requirement |
|---|---|
| FR1 | The tool shall load a subscriber master list from a JSON file. |
| FR2 | The tool shall load a batch of CDRs from a CSV file. |
| FR3 | The tool shall validate each CDR row and skip (logging + counting) malformed rows without crashing the batch. |
| FR4 | The tool shall compute a per-call cost based on call type (`domestic` / `roaming` / `international`) and duration. |
| FR5 | The tool shall flag a call as fraud-suspect if: (a) duration > 0 but computed cost == 0, or (b) it's `international` and duration exceeds a configurable threshold (default 3600s). |
| FR6 | The tool shall aggregate total cost and call count per subscriber. |
| FR7 | The tool shall identify CDRs referencing an `msisdn` not present in the subscriber master list, and report them separately as "unknown subscriber" anomalies. |
| FR8 | The tool shall write a daily report as indented JSON, and print a human-readable console summary. |
| FR9 | Input file paths and the fraud duration threshold shall be configurable via CLI arguments, not hardcoded. |

### Non-Functional Requirements

| ID | Requirement |
|---|---|
| NFR1 | All status/error events must be logged with timestamps and severity levels — no bare `print` for this. |
| NFR2 | If more than 10% of CDR rows in a batch are malformed, the tool must log a `CRITICAL` message and exit with a non-zero exit code instead of writing a partial report. |
| NFR3 | Code must be organized into separate modules (parsing/rating/fraud/reporting/IO), not one script. |
| NFR4 | All monetary values must be rounded to 2 decimal places. |

**Acceptance criteria.** Running the tool against the provided sample `subscribers.json` and
`cdrs.csv` (generated below) produces a `report.json` whose per-subscriber totals you can verify by
hand-calculating at least two subscribers' costs yourself.


## 1.2 Architecture (Simple)

**Module layout:**
```
telecom_cli/
    main.py            # CLI entry point (argparse), orchestrates everything below
    config.py            # RATES dict, default thresholds — the ONLY place magic numbers live
    io_utils.py            # load_subscribers(), load_cdrs(), write_report()
    models.py                # Subscriber class, (optionally) CDR class
    rating.py                  # compute_cost()
    fraud.py                     # is_suspicious()
    reporting.py                   # build_report()
```

**Data flow:**
```
 subscribers.json ──┐
                     ├──▶ main.py ──▶ parse & validate ──▶ rate each CDR ──▶ fraud-check
 cdrs.csv ───────────┘                                                          │
                                                                                 ▼
                                                              aggregate per subscriber
                                                                                 │
                                                                                 ▼
                                                          report.json  +  console summary
```

**Layered view** (same shape you'll see again in the FastAPI project later in the curriculum):

| Layer | Responsibility | Modules |
|---|---|---|
| CLI Layer | Parse arguments, wire everything together, set exit code | `main.py` |
| Business Logic Layer | Rating, fraud rules, aggregation | `rating.py`, `fraud.py`, `reporting.py` |
| Data Access Layer | Reading/writing files | `io_utils.py` |


## 1.3 Sample Data

Run the cell below to generate `subscribers.json` and `cdrs.csv` in your working directory —
deliberately includes a few "bad" rows so you have something real to validate against.


In [1]:
import json
import csv

subscribers = [
    {"msisdn": "9876543210", "plan_type": "postpaid"},
    {"msisdn": "9812345678", "plan_type": "prepaid"},
    {"msisdn": "9800011122", "plan_type": "postpaid"},
]
with open("subscribers.json", "w") as f:
    json.dump(subscribers, f, indent=2)

cdr_rows = [
    {"msisdn": "9876543210", "call_type": "domestic", "duration_sec": "180"},
    {"msisdn": "9876543210", "call_type": "international", "duration_sec": "4200"},
    {"msisdn": "9812345678", "call_type": "roaming", "duration_sec": "60"},
    {"msisdn": "9812345678", "call_type": "domestic", "duration_sec": "0"},
    {"msisdn": "9800011122", "call_type": "international", "duration_sec": "300"},
    {"msisdn": "9999999999", "call_type": "domestic", "duration_sec": "120"},   # unknown subscriber
    {"msisdn": "9800011122", "call_type": "domestic", "duration_sec": "notanumber"},  # malformed
]
with open("cdrs.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["msisdn", "call_type", "duration_sec"])
    writer.writeheader()
    writer.writerows(cdr_rows)

print("Sample data written: subscribers.json, cdrs.csv")


Sample data written: subscribers.json, cdrs.csv


## 1.4 Milestone 1 — Setup, Syntax, Variables & Data Types, Control Flow & Loops

**Q1.1** Design the folder/module layout from section 1.2 (in this notebook environment, module
files aren't required — write your functions in the code cells below as if each comment marked a
separate file). Write a startup banner print using an f-string that includes today's date
(`datetime.date.today()`), guarded so it only prints when the "script" is run directly.

**Q1.2** Write `parse_cdr_line(row: dict) -> dict` that takes one CSV row (already a dict from
`csv.DictReader`) and returns a cleaned dict with `duration_sec` cast to `int`. Raise `ValueError`
with a clear message if `duration_sec` isn't a valid integer or a required field is missing.

**Q1.3** Write `compute_cost(call_type: str, duration_sec: int) -> float`. For THIS milestone,
implement the rate table (domestic 1.5, roaming 8.0, international 12.0 per minute) using
`if/elif/else` — you'll refactor this to a dict lookup in Milestone 2.

**Q1.4** Using a `for` loop (no comprehensions yet), iterate over a hardcoded list of at least 4
sample CDR dicts and print each one's computed cost.

**Q1.5 (Tricky)** Using `while`, write a small retry loop that simulates re-attempting a "flaky"
file read up to 3 times before giving up and logging failure (same shape as the retry pattern from
the Loops session — no need for real file flakiness, just simulate it with a counter).


In [ ]:
# Q1.1


In [ ]:
# Q1.2


In [ ]:
# Q1.3


In [ ]:
# Q1.4


In [ ]:
# Q1.5


## 1.5 Milestone 2 — Data Structures, Functions & Basic OOP, String Manipulation

**Q2.1** Refactor `compute_cost`'s rate table into a module-level `RATES` dict and rewrite the
function to use `.get(call_type, RATES["domestic"])` instead of `if/elif/else`.

**Q2.2** Define a `Subscriber` class with `msisdn`, `plan_type`, and a `calls` list (of CDR dicts).
Give it a method `total_cost(self) -> float` summing the cost of every call in `self.calls`.

**Q2.3** Define a `CDR` class representing one call record, with a method `is_suspicious(self) ->
bool` implementing the two fraud rules from FR5. (You may keep using plain dicts for CDRs elsewhere
if you prefer — but this class must exist and be used somewhere in your fraud-checking logic.)

**Q2.4** Using a `set`, compute the set of unique `msisdn` values that appear in the CDR batch but
do NOT exist in the subscriber master list — these are the FR7 "unknown subscriber" anomalies.

**Q2.5 (String manipulation)** Some legacy CDR sources deliver a single pipe-delimited string
instead of a proper CSV row, e.g. `"9876543210|domestic|180"`. Write `parse_legacy_line(line: str)
-> dict` using `.split("|")` and `.strip()` that normalizes this into the same shape
`parse_cdr_line` produces.

**Q2.6 (Tricky — answer in a markdown cell or a comment)** Why is a `set` the right choice for the
"unknown subscribers" collection in Q2.4, rather than a `list`? What would go wrong with a list if
the same unknown `msisdn` appeared in 50 CDRs in one batch?


In [ ]:
# Q2.1


In [ ]:
# Q2.2


In [ ]:
# Q2.3


In [ ]:
# Q2.4


In [ ]:
# Q2.5


In [ ]:
# Q2.6 -- write your answer as a comment here


## 1.6 Milestone 3 — File Handling (CSV/JSON) & Scripting Best Practices

**Q3.1** Write `load_subscribers(json_path: str) -> dict` that reads the JSON file and returns a
dict keyed by `msisdn`, valued by `Subscriber` instances.

**Q3.2** Write `load_cdrs(csv_path: str) -> list` using `csv.DictReader`, applying
`parse_cdr_line` to each row inside a `try/except ValueError`, logging a `WARNING` and skipping
(not crashing on) any malformed row. Track how many rows were skipped.

**Q3.3** Write `write_report(report: dict, output_path: str) -> None` that writes the report as
indented JSON (`json.dump(..., indent=2)`).

**Q3.4** Wire everything together in a `main()` function using `argparse` with `--subscribers`,
`--cdrs`, `--output`, and `--fraud-threshold-sec` (default `3600`) arguments. Configure `logging`
once, at the very start of `main()`.

**Q3.5** Implement NFR2: if more than 10% of CDR rows in the batch were malformed (from Q3.2's
count), log `CRITICAL` and call `sys.exit(1)` instead of writing a report.

**Q3.6 (Tricky — answer in a markdown cell or comment)** A teammate suggests wrapping all of
`load_cdrs` in a single broad `except Exception:` "just to be safe." Explain why this is worse than
catching `ValueError` specifically, referencing what a bare/broad except would hide from you if,
say, the CSV file itself didn't exist at all.


In [ ]:
# Q3.1


In [ ]:
# Q3.2


In [ ]:
# Q3.3


In [ ]:
# Q3.4 -- remember argparse needs an explicit args list to test inside a notebook, e.g. parser.parse_args(['--subscribers', 'subscribers.json', ...])


In [ ]:
# Q3.5


In [ ]:
# Q3.6 -- write your answer as a comment here


---
# Project 2 — Retail Inventory & Sales Reconciliation CLI

**Client:** Northfield Retail Co. (fictitious mid-size retail chain)
**Requested by:** Finance & Store Operations, via the Engineering Lead


## 2.1 Problem Statement / PRD

**Background.** Each store emails a daily sales export (CSV) to HQ. Finance currently reconciles
this by hand against the inventory system's snapshot (JSON) to catch stock issues and compute
revenue — a slow, manual, spreadsheet-driven process. Engineering has scoped a CLI tool to automate
daily reconciliation for a single store's batch. You are the engineer assigned to build it.

**Goals**
- Automate ingestion and validation of daily sales + inventory data.
- Compute revenue per transaction, correctly applying discounts.
- Flag stock issues: oversold SKUs and low-stock SKUs.
- Produce a daily reconciliation report, machine-readable (JSON) and human-readable (console).

**Out of scope (Phase 2, not this project)**
- No live POS/database integration — file-based batch processing only.
- No multi-day trend analysis or forecasting.
- No user interface.

**Stakeholders**
- *Finance Analyst* — consumes the daily report for revenue reconciliation.
- *Store Ops Manager* — needs the stock-issue flags to act on same-day.

### Functional Requirements

| ID | Requirement |
|---|---|
| FR1 | The tool shall load a product inventory snapshot from a JSON file (`sku`, `name`, `unit_price`, `stock_qty`, `category`). |
| FR2 | The tool shall load a batch of sales transactions from a CSV file (`sku`, `qty_sold`, `discount_pct`, `channel`). |
| FR3 | The tool shall validate each sales row and skip (logging + counting) malformed rows without crashing the batch. |
| FR4 | The tool shall compute line revenue as `qty_sold * unit_price * (1 - discount_pct/100)`, rounded to 2 decimal places. |
| FR5 | The tool shall flag a SKU as **oversold** if total `qty_sold` across the batch exceeds its `stock_qty`. |
| FR6 | The tool shall flag a SKU as **low-stock** if its remaining stock (`stock_qty` - total `qty_sold`) falls below a configurable reorder threshold (default 10), using a set to track unique low-stock SKUs. |
| FR7 | The tool shall identify sales rows referencing a `sku` not present in inventory, and report them separately as "unknown SKU" anomalies. |
| FR8 | The tool shall aggregate total revenue by `category` and by `channel` (`online` / `in-store`). |
| FR9 | The tool shall write a daily report as indented JSON, and print a human-readable console summary. |
| FR10 | Input file paths and the low-stock reorder threshold shall be configurable via CLI arguments. |

### Non-Functional Requirements

| ID | Requirement |
|---|---|
| NFR1 | All status/error events must be logged with timestamps and severity levels — no bare `print` for this. |
| NFR2 | If more than 10% of sales rows in a batch are malformed, the tool must log `CRITICAL` and exit with a non-zero exit code instead of writing a partial report. |
| NFR3 | Code must be organized into separate modules (parsing/calculations/reporting/IO), not one script. |
| NFR4 | All monetary values must be rounded to 2 decimal places. |

**Acceptance criteria.** Running the tool against the provided sample `inventory.json` and
`sales.csv` (generated below) produces a `report.json` whose revenue totals and stock flags you can
verify by hand-calculating at least two SKUs yourself.


## 2.2 Architecture (Simple)

**Module layout:**
```
retail_cli/
    main.py            # CLI entry point (argparse), orchestrates everything below
    config.py            # default reorder threshold — the ONLY place magic numbers live
    io_utils.py            # load_inventory(), load_sales(), write_report()
    models.py                # Product class, SaleTransaction class
    calculations.py            # compute_revenue(), stock flag rules
    reporting.py                 # build_report()
```

**Data flow:**
```
 inventory.json ──┐
                   ├──▶ main.py ──▶ parse & validate ──▶ compute revenue ──▶ stock-flag check
 sales.csv ────────┘                                                              │
                                                                                   ▼
                                                                aggregate by category/channel
                                                                                   │
                                                                                   ▼
                                                            report.json  +  console summary
```

**Layered view:**

| Layer | Responsibility | Modules |
|---|---|---|
| CLI Layer | Parse arguments, wire everything together, set exit code | `main.py` |
| Business Logic Layer | Revenue calculation, stock-flag rules, aggregation | `calculations.py`, `reporting.py` |
| Data Access Layer | Reading/writing files | `io_utils.py` |


## 2.3 Sample Data

Run the cell below to generate `inventory.json` and `sales.csv` — includes an oversold SKU, a
low-stock SKU, an unknown SKU, and a malformed row.


In [ ]:
import json
import csv

inventory = [
    {"sku": "SKU-1001", "name": "Wireless Mouse", "unit_price": 19.99, "stock_qty": 50, "category": "Electronics"},
    {"sku": "SKU-1002", "name": "Bluetooth Speaker", "unit_price": 45.50, "stock_qty": 8, "category": "Electronics"},
    {"sku": "SKU-2001", "name": "Notebook Pack", "unit_price": 5.25, "stock_qty": 200, "category": "Stationery"},
]
with open("inventory.json", "w") as f:
    json.dump(inventory, f, indent=2)

sales_rows = [
    {"sku": "SKU-1001", "qty_sold": "5", "discount_pct": "10", "channel": "online"},
    {"sku": "SKU-1002", "qty_sold": "10", "discount_pct": "0", "channel": "in-store"},   # oversold: stock is 8
    {"sku": "SKU-2001", "qty_sold": "20", "discount_pct": "5", "channel": "online"},
    {"sku": "SKU-9999", "qty_sold": "3", "discount_pct": "0", "channel": "online"},        # unknown SKU
    {"sku": "", "qty_sold": "4", "discount_pct": "0", "channel": "in-store"},                # malformed: missing sku
]
with open("sales.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["sku", "qty_sold", "discount_pct", "channel"])
    writer.writeheader()
    writer.writerows(sales_rows)

print("Sample data written: inventory.json, sales.csv")


## 2.4 Milestone 1 — Setup, Syntax, Variables & Data Types, Control Flow & Loops

**Q1.1** Write a startup banner print using an f-string that includes today's date, guarded so it
only runs when the "script" is executed directly.

**Q1.2** Write `parse_sale_row(row: dict) -> dict` that casts `qty_sold` to `int` and
`discount_pct` to `float`, raising `ValueError` with a clear message on bad data or a missing `sku`.

**Q1.3** Write `compute_revenue(unit_price: float, qty_sold: int, discount_pct: float) -> float`.
Using `if` control flow, clamp `discount_pct` to the range 0–100 if it falls outside that range
(treat an invalid value as 0 rather than letting it silently produce a negative or inflated revenue).

**Q1.4** Using a `for` loop (no comprehensions yet), iterate over a hardcoded list of at least 4
sample sale dicts and print each one's computed revenue.

**Q1.5 (Tricky)** Using `while`, simulate a retry loop for re-attempting a "flaky" file read up to
3 times before giving up and logging failure.


In [ ]:
# Q1.1


In [ ]:
# Q1.2


In [ ]:
# Q1.3


In [ ]:
# Q1.4


In [ ]:
# Q1.5


## 2.5 Milestone 2 — Data Structures, Functions & Basic OOP, String Manipulation

**Q2.1** Build `revenue_by_category: dict` and `revenue_by_channel: dict` that accumulate revenue
as you process each sale (FR8).

**Q2.2** Define a `Product` class (`sku`, `name`, `unit_price`, `stock_qty`, `category`) with a
method `remaining_stock(self, qty_sold: int) -> int`.

**Q2.3** Define a `SaleTransaction` class with a method `line_revenue(self, product) -> float`
that reuses (or reimplements) Milestone 1's `compute_revenue` logic.

**Q2.4** Using a `set`, compute the set of SKUs sold that do NOT exist in inventory (FR7's unknown
SKU anomalies), and separately, the set of SKUs that are low-stock after this batch (FR6).

**Q2.5 (String manipulation)** SKUs follow the pattern `"SKU-<category_code><number>"`, e.g.
`"SKU-1001"`, where the first digit after the dash indicates category (`1` = Electronics, `2` =
Stationery). Write `category_code_from_sku(sku: str) -> str` that extracts this code using string
slicing — purely from the SKU string, as a sanity cross-check against the inventory's declared
`category` field.

**Q2.6 (Tricky — answer in a markdown cell or comment)** Why is a `set` the right choice for
tracking low-stock SKUs across a batch with potentially many rows for the same SKU, rather than a
`list`? What specific bug would a list introduce here?


In [ ]:
# Q2.1


In [ ]:
# Q2.2


In [ ]:
# Q2.3


In [ ]:
# Q2.4


In [ ]:
# Q2.5


In [ ]:
# Q2.6 -- write your answer as a comment here


## 2.6 Milestone 3 — File Handling (CSV/JSON) & Scripting Best Practices

**Q3.1** Write `load_inventory(json_path: str) -> dict` returning a dict keyed by `sku`, valued by
`Product` instances.

**Q3.2** Write `load_sales(csv_path: str) -> list` using `csv.DictReader`, applying
`parse_sale_row` to each row inside a `try/except ValueError`, logging a `WARNING` and skipping
malformed rows. Track the count of skipped rows.

**Q3.3** Write `write_report(report: dict, output_path: str) -> None` writing indented JSON.

**Q3.4** Wire everything together in a `main()` function using `argparse` with `--inventory`,
`--sales`, `--output`, and `--reorder-threshold` (default `10`) arguments. Configure `logging` once
at the start of `main()`.

**Q3.5** Implement NFR2: if more than 10% of sales rows were malformed, log `CRITICAL` and
`sys.exit(1)` instead of writing a report.

**Q3.6 (Tricky — answer in a markdown cell or comment)** Same question as Project 1's Q3.6, applied
here: why is catching `ValueError` specifically in `load_sales` better than a broad
`except Exception:`? What would a broad except hide if `sales.csv` had the wrong file encoding
entirely, versus just one bad row?


In [ ]:
# Q3.1


In [ ]:
# Q3.2


In [ ]:
# Q3.3


In [ ]:
# Q3.4 -- remember argparse needs an explicit args list to test inside a notebook


In [ ]:
# Q3.5


In [ ]:
# Q3.6 -- write your answer as a comment here


---
## Deliverables Checklist (Both Projects)

- [ ] Package structure matches the Architecture section — no single monolithic script.
- [ ] A `README.md` with setup and run instructions.
- [ ] Running `main.py` end-to-end against the provided sample data produces a correct `report.json`.
- [ ] A human-readable console summary prints on run.
- [ ] `logging` used throughout for status/errors — no `print` outside the final summary.
- [ ] Malformed-row handling demonstrated — the batch doesn't crash on bad rows.
- [ ] Exit-code behavior implemented (NFR2) — verify it by deliberately feeding a batch with >10% bad rows.
- [ ] Code follows the shared Coding Standards section (docstrings, naming, no bare except, no magic numbers).

## Evaluation Rubric (Both Projects)

| Criteria | Weight | What "Good" Looks Like |
|---|---|---|
| Correctness | 40% | Report totals match hand-calculated expected values on the sample data. |
| Code Quality & Standards | 25% | PEP8-clean, docstrings present, naming conventions followed, modular structure matching the architecture. |
| Error Handling & Robustness | 20% | Malformed rows logged and skipped (not crashed on), correct exit-code behavior, no bare/broad except. |
| Documentation | 15% | A README clear enough that a teammate could run the tool without asking you questions. |

---

### Closing note

Both projects deliberately reuse the same shape: load → validate → compute → flag → aggregate →
report. That's not an accident — it's the same shape you'll see again, at larger scale, in the
FastAPI capstone project later in the curriculum. Getting comfortable with this shape now, using
nothing but core Python, pays off directly later.
